In [10]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import warnings
import joblib
import numpy as np
import pandas as pd

kaggle_test_df = pd.read_csv(r'..\..\..\data\003_kaggle_competition_data\25261_classification_sample_submission.csv')
kaggle_smiles = kaggle_test_df[['smiles']].copy()

In [11]:


# Load 
final_model = joblib.load(r'algor_used\classification_lr_model.joblib')
scaler = joblib.load(r'algor_used\classification_scaler.joblib')
feature_cols = joblib.load(r'algor_used\classification_feature_names.joblib')
le = joblib.load(r'algor_used\label_encoder_classification.joblib')
print(f"Loaded: feature_names ({len(feature_cols)} features)")

Loaded: feature_names (228 features)


In [12]:
descriptor_names = [desc[0] for desc in Descriptors._descList]
descriptor_calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

def compute_descriptors_fast(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    try:
        return list(descriptor_calculator.CalcDescriptors(mol))
    except:
        return [np.nan] * len(descriptor_names)

test_descriptors = []
for idx, smiles in enumerate(kaggle_test_df['smiles']):
    if idx % 50 == 0:
        print(f"   Progress: {idx}/{len(kaggle_test_df)}...")
    test_descriptors.append(compute_descriptors_fast(smiles))

X_kaggle = pd.DataFrame(test_descriptors, columns=descriptor_names)
print(f"  Descriptors computed: {X_kaggle.shape}")

   Progress: 0/200...
   Progress: 50/200...
   Progress: 100/200...
   Progress: 150/200...
  Descriptors computed: (200, 217)


In [13]:
def extract_bond_atom_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {f: 0 for f in ['n_bonds', 'n_single_bonds', 'n_double_bonds',
                'n_triple_bonds', 'n_aromatic_bonds', 'count_c', 'count_h',
                'count_o', 'count_n', 'count_s', 'count_f', 'count_cl',
                'count_br', 'count_i', 'count_p', 'count_si']}
    
    bonds = mol.GetBonds()
    atoms = mol.GetAtoms()
    atom_symbols = [atom.GetSymbol() for atom in atoms]
    
    return {
        'n_bonds': len(bonds),
        'n_single_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 1.0),
        'n_double_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 2.0),
        'n_triple_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 3.0),
        'n_aromatic_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 1.5),
        'count_c': atom_symbols.count('C'),
        'count_h': atom_symbols.count('H'),
        'count_o': atom_symbols.count('O'),
        'count_n': atom_symbols.count('N'),
        'count_s': atom_symbols.count('S'),
        'count_f': atom_symbols.count('F'),
        'count_cl': atom_symbols.count('Cl'),
        'count_br': atom_symbols.count('Br'),
        'count_i': atom_symbols.count('I'),
        'count_p': atom_symbols.count('P'),
        'count_si': atom_symbols.count('Si')
    }

In [14]:

bond_features = [extract_bond_atom_features(s) for s in kaggle_test_df['smiles']]
bond_df = pd.DataFrame(bond_features)

X_kaggle = pd.concat([X_kaggle, bond_df], axis=1)
X_kaggle = X_kaggle.loc[:, ~X_kaggle.columns.duplicated()]

print(f"   Bond/atom features: {bond_df.shape}")
print(f"   Combined: {X_kaggle.shape}")

   Bond/atom features: (200, 16)
   Combined: (200, 233)


In [15]:
X_kaggle['TPSA_x_MolLogP'] = X_kaggle['TPSA'] * X_kaggle['MolLogP']
X_kaggle['aromatic_x_hetero'] = X_kaggle['NumAromaticRings'] * X_kaggle['NumHeteroatoms']
X_kaggle['bonds_x_carbon'] = X_kaggle['n_bonds'] * X_kaggle['count_c']
X_kaggle['aromatic_bonds_x_carbon'] = X_kaggle['n_aromatic_bonds'] * X_kaggle['count_c']

X_kaggle['pi_electrons'] = X_kaggle['n_aromatic_bonds'] * 2 + X_kaggle['n_double_bonds'] * 2
X_kaggle['conjugation_score'] = (X_kaggle['NumAromaticRings'] * X_kaggle['n_aromatic_bonds']) / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['aromatic_ratio'] = X_kaggle['NumAromaticRings'] / (X_kaggle['RingCount'] + 1)
X_kaggle['aromatic_atoms_ratio'] = (X_kaggle['n_aromatic_bonds'] * 2) / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['aromatic_x_double'] = X_kaggle['n_aromatic_bonds'] * X_kaggle['n_double_bonds']

# Heteroatom features (5)
X_kaggle['hetero_ratio'] = X_kaggle['NumHeteroatoms'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['hetero_in_aromatic'] = X_kaggle['NumAromaticHeterocycles'] / (X_kaggle['NumAromaticRings'] + 1)
X_kaggle['nitrogen_ratio'] = X_kaggle['count_n'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['oxygen_ratio'] = X_kaggle['count_o'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['sulfur_ratio'] = X_kaggle['count_s'] / (X_kaggle['HeavyAtomCount'] + 1)

# EWG/EDG features (5)
X_kaggle['EWG_count'] = X_kaggle['count_f'] + X_kaggle['count_cl'] + X_kaggle['count_br'] + X_kaggle['fr_nitro'] + X_kaggle['fr_nitro_arom']
X_kaggle['EWG_density'] = X_kaggle['EWG_count'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['EDG_count'] = X_kaggle['fr_NH2'] + X_kaggle['fr_NH1'] + X_kaggle['fr_NH0'] + X_kaggle['fr_methoxy'] + X_kaggle['fr_Ar_NH']
X_kaggle['EDG_density'] = X_kaggle['EDG_count'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['EWG_EDG_balance'] = (X_kaggle['EDG_count'] - X_kaggle['EWG_count']) / (X_kaggle['HeavyAtomCount'] + 1)

# Size-normalized properties (4)
X_kaggle['TPSA_per_atom'] = X_kaggle['TPSA'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['TPSA_per_MW'] = X_kaggle['TPSA'] / (X_kaggle['MolWt'] + 1)
X_kaggle['MolMR_per_atom'] = X_kaggle['MolMR'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['LogP_per_atom'] = X_kaggle['MolLogP'] / (X_kaggle['HeavyAtomCount'] + 1)

# Charge distribution (3)
X_kaggle['charge_range'] = X_kaggle['MaxPartialCharge'] - X_kaggle['MinPartialCharge']
X_kaggle['charge_extent'] = X_kaggle['MaxAbsPartialCharge'] - X_kaggle['MinAbsPartialCharge']
X_kaggle['charge_asymmetry'] = abs(X_kaggle['MaxPartialCharge'] + X_kaggle['MinPartialCharge'])

# Complexity/rigidity (4)
X_kaggle['unsaturation'] = (X_kaggle['n_double_bonds'] + X_kaggle['n_triple_bonds'] + X_kaggle['n_aromatic_bonds']) / (X_kaggle['n_bonds'] + 1)
X_kaggle['rigidity'] = X_kaggle['RingCount'] / (X_kaggle['NumRotatableBonds'] + 1)
X_kaggle['ring_complexity'] = (X_kaggle['NumAromaticRings'] + X_kaggle['NumSaturatedRings'] + X_kaggle['NumAliphaticRings']) / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['heterocycle_ratio'] = X_kaggle['NumHeterocycles'] / (X_kaggle['RingCount'] + 1)

# Orbital-specific interactions (6)
X_kaggle['aromatic_polarity'] = X_kaggle['NumAromaticRings'] * X_kaggle['TPSA']
X_kaggle['hetero_conjugation'] = X_kaggle['NumHeteroatoms'] * X_kaggle['n_aromatic_bonds']
X_kaggle['nitrogen_aromatic'] = X_kaggle['count_n'] * X_kaggle['NumAromaticRings']
X_kaggle['oxygen_aromatic'] = X_kaggle['count_o'] * X_kaggle['NumAromaticRings']
X_kaggle['hetero_per_ring'] = X_kaggle['NumHeteroatoms'] / (X_kaggle['RingCount'] + 1)
X_kaggle['aromatic_hetero_ratio'] = X_kaggle['NumAromaticHeterocycles'] / (X_kaggle['NumHeteroatoms'] + 1)

# Bonding character (3)
X_kaggle['single_bond_ratio'] = X_kaggle['n_single_bonds'] / (X_kaggle['n_bonds'] + 1)
X_kaggle['multiple_bond_ratio'] = (X_kaggle['n_double_bonds'] + X_kaggle['n_triple_bonds']) / (X_kaggle['n_bonds'] + 1)
X_kaggle['aromatic_bond_ratio'] = X_kaggle['n_aromatic_bonds'] / (X_kaggle['n_bonds'] + 1)

X_kaggle = X_kaggle.fillna(0)

print(f"   Added 35 chemistry features to Kaggle data")
print(f"   Total: {X_kaggle.shape}")

   Added 35 chemistry features to Kaggle data
   Total: (200, 272)


In [16]:
X_kaggle = X_kaggle.replace([np.inf, -np.inf], 0)

# Clip extremes
for col in X_kaggle.select_dtypes(include=[np.number]).columns:
    q99 = X_kaggle[col].quantile(0.99)
    q01 = X_kaggle[col].quantile(0.01)
    if q99 > q01:
        X_kaggle[col] = X_kaggle[col].clip(lower=q01, upper=q99)

# DON'T remove constant features!
print(f"   After preprocessing: {X_kaggle.shape}")

   After preprocessing: (200, 272)


In [17]:
missing_features = [feat for feat in feature_cols if feat not in X_kaggle.columns]

if missing_features:
    print(f"Adding {len(missing_features)} missing features as zeros")
    for feat in missing_features:
        X_kaggle[feat] = 0
else:
    print(f"All {len(feature_cols)} features present!")

X_kaggle = X_kaggle[feature_cols]

print(f"   Final shape: {X_kaggle.shape}")
assert X_kaggle.shape[1] == len(feature_cols), "Feature count mismatch!"

X_kaggle_scaled = scaler.transform(X_kaggle)
print(f"Scaled: {X_kaggle_scaled.shape}")

All 228 features present!
   Final shape: (200, 228)
Scaled: (200, 228)


In [18]:
predictions_encoded = final_model.predict(X_kaggle_scaled)
predictions = le.inverse_transform(predictions_encoded)

print(f"Predictions: {len(predictions)}")

# Distribution
pred_counts = pd.Series(predictions).value_counts()
print(f"\n   Prediction distribution:")
for cls in ['high', 'low', 'medium']:
    count = pred_counts.get(cls, 0)
    pct = count / len(predictions) * 100

submission = pd.DataFrame({
    'smiles': kaggle_test_df['smiles'].values,
    'stability': predictions
})

submission.to_csv(r'algor_used\kaggle_submission_ensemble.csv', index=False)

print(f" Saved: kaggle_submission_ensemble.csv")
print(f"\n   First 10 predictions:")
print(submission.head(10).to_string(index=False))

Predictions: 200

   Prediction distribution:
 Saved: kaggle_submission_ensemble.csv

   First 10 predictions:
                                                       smiles stability
[SiH2]1cccc1-c1ccc(-c2cnc(-c3nccc4nsnc34)c3nsnc23)c2c[nH]cc12       low
      c1[nH]cc2c1c(cc1c3ccccc3c3c4ccccc4oc3c21)-c1nccc2nsnc12      high
   c1cc2c(scc2o1)-c1ccc(o1)-c1ccc(-c2cccc3c[nH]cc23)c2ccccc12      high
       c1cc2csc(-c3cc4c(s3)c3cnc5ccc6ccccc6c5c3c3ccccc43)c2s1      high
                             C1cccc1-c1ccc(-c2ccccc2)c2nsnc12    medium
                 C1C=c2c3cc[se]c3c3sc4C=C(Cc4c3c2=C1)c1ccccn1      high
          c1[nH]cc2c1ccc1sc3c4ccc(cc4[se]c3c21)-c1cncc2nsnc12      high
                      c1c[nH]c(c1)-c1cc2sc3c4ncccc4[se]c3c2s1      high
    c1cc2csc(-c3ccc([nH]3)-c3cnc(-c4nccc5nsnc45)c4nsnc34)c2o1       low
              C1c2cc(-c3ncncn3)c3c[nH]cc3c2-c2[nH]c3cccnc3c12       low
